# Make DBs from data in SQLite

Due to size of the data, we focus on romance genre only 

In [1]:
import sqlite3
import json
from tqdm import tqdm
import os
import gzip
import numpy as np
import pandas as pd

In [2]:
# open romance.db in sqlite3
conn = sqlite3.connect("byGenre/romance.db")
cursor = conn.cursor()

In [3]:
# key reinforcement
cursor.execute("PRAGMA foreign_keys = ON;")

# speed boost
cursor.execute("PRAGMA journal_mode = WAL;")
cursor.execute("PRAGMA synchronous = OFF;")
cursor.execute("PRAGMA temp_store = FILE;")
cursor.execute("PRAGMA cache_size = -200000;")  # ~200MB RAM

## Design db Schema

### romance.db

#### Core Tables:
├ books  
├ authors  
├ shelves  
├ users (derived)  

#### Link Tables:
├ book_authors  
├ book_shelves  
├ similar_books  

#### Transaction Tables:
├ reviews  
├ ratings  

## Create tables

In [ ]:
# create books table 
cursor.execute("""
CREATE TABLE books (
    book_id TEXT PRIMARY KEY NOT NULL,
    title TEXT,
    description TEXT,
    average_rating REAL,
    ratings_count INTEGER,
    text_reviews_count INTEGER,
    num_pages INTEGER,
    publisher TEXT,
    publication_year INTEGER,
    publication_month INTEGER,
    publication_day INTEGER
    
)
""")

# create authors table 
cursor.execute("""
CREATE TABLE authors (
    author_id TEXT PRIMARY KEY NOT NULL,
    author_name TEXT NOT NULL,
    author_average_rating REAL,
    author_ratings_count INTEGER
    
)
""")

# create book_authors table 
cursor.execute("""
CREATE TABLE book_authors (
    book_id TEXT NOT NULL,
    author_id TEXT NOT NULL,
    PRIMARY KEY (book_id, author_id),
    FOREIGN KEY (book_id) REFERENCES books(book_id),
    FOREIGN KEY (author_id) REFERENCES authors(author_id)
)
""")

# create shelves table 
cursor.execute("""
CREATE TABLE shelves (
    shelf_id INTEGER PRIMARY KEY AUTOINCREMENT,
    shelf_name TEXT UNIQUE NOT NULL
)
""")

# create book_shelves table 
cursor.execute("""
CREATE TABLE book_shelves (
    book_id TEXT NOT NULL,
    shelf_id INTEGER NOT NULL,
    PRIMARY KEY (book_id, shelf_id),
    FOREIGN KEY (book_id) REFERENCES books(book_id),
    FOREIGN KEY (shelf_id) REFERENCES shelves(shelf_id)
)
""")

# create similar_books table 
cursor.execute("""
CREATE TABLE similar_books (
    book_id TEXT NOT NULL,
    similar_book_id TEXT NOT NULL,
    PRIMARY KEY (book_id, similar_book_id),
    FOREIGN KEY (book_id) REFERENCES books(book_id)
)
""")

# create reviews table 
cursor.execute("""
CREATE TABLE reviews (
    review_id TEXT PRIMARY KEY,
    user_id TEXT NOT NULL,
    book_id TEXT NOT NULL,
    rating REAL,
    review_text TEXT,
    FOREIGN KEY (book_id) REFERENCES books(book_id)
)
""")

In [27]:
# I had to re-do the ratings table so I deleted and recreated it
cursor.execute(f"DROP TABLE IF EXISTS ratings;")
# create ratings table 
cursor.execute("""
CREATE TABLE ratings (
    review_id TEXT PRIMARY KEY NOT NULL,
    user_id TEXT NOT NULL,
    book_id TEXT NOT NULL,
    rating REAL,
    FOREIGN KEY (book_id) REFERENCES books(book_id)
)
""")
conn.commit()

## Define Intake Functions

In [16]:
def safe_int(x):
    return int(x) if x and x.isdigit() else None

def safe_float(x):
    try:
        return float(x)
    except:
        return None

In [38]:
def load_json_authors(path):
    data = []
    with gzip.open(path, "rt") as f:
        for i, line in enumerate(tqdm(f)):
            auth = json.loads(line)
    
            cursor.execute("""
                INSERT OR IGNORE INTO authors (
                author_id, author_name, author_average_rating, author_ratings_count
                )
                VALUES (?, ?, ?, ?)
            """, (
                auth['author_id'],
                auth["name"],
                safe_float(auth["average_rating"]),
                safe_int(auth['ratings_count'])
            ))
            if i % 5000 == 0 and i > 0:
                conn.commit()
    conn.commit()

In [5]:
# Functions for loading books info into the database 
def load_json_books(path):
    with gzip.open(path, "rt") as f:
        for i, line in enumerate(tqdm(f)):
            book = json.loads(line)

            # populate books table
            cursor.execute("""
                INSERT OR IGNORE INTO books (
                book_id, title, description, average_rating,
                ratings_count, text_reviews_count, num_pages,
                publisher, publication_year, publication_month, publication_day
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                book["book_id"],
                book["title"],
                book['description'],
                safe_float(book["average_rating"]),
                safe_int(book["ratings_count"]),
                safe_int(book['text_reviews_count']),
                safe_int(book['num_pages']),
                book['publisher'],
                safe_int(book['publication_year']),
                safe_int(book['publication_month']),
                safe_int(book['publication_day'])
            ))

            # populate book_authors
            for author in book['authors']:
                cursor.execute("""
                    INSERT OR IGNORE INTO book_authors (
                    book_id, author_id
                    )
                    VALUES (?, ?)
                """, (
                    book["book_id"],
                    author["author_id"]
                ))

            # populate shelves and book_shelves
            for shelf in book['popular_shelves']:
                
                # Insert shelf if new
                cursor.execute("""
                    INSERT OR IGNORE INTO shelves (shelf_name)
                    VALUES (?)
                """, (shelf['name'],))
            
                # Retrieve the shelf_id
                cursor.execute("""
                    SELECT shelf_id FROM shelves
                    WHERE shelf_name = ?
                """, (shelf['name'],))
            
                shelf_id = cursor.fetchone()[0]
            
                # Insert into junction table
                cursor.execute("""
                    INSERT OR IGNORE INTO book_shelves (book_id, shelf_id)
                    VALUES (?, ?)
                """, (
                    book["book_id"],
                    shelf_id
                ))

            for similar_book in book['similar_books']:
                cursor.execute("""
                    INSERT OR IGNORE INTO similar_books (
                    book_id, similar_book_id
                    )
                    VALUES (?, ?)
                """, (
                    book["book_id"],
                    similar_book
                ))

                
            if i % 5000 == 0 and i > 0:
                conn.commit()
    conn.commit()

In [ ]:
# Functions for loading reviews into the database 
def load_json_reviews(path):
    data = []
    with gzip.open(path, "rt") as f:
        for i, line in enumerate(tqdm(f)):
            rev = json.loads(line)
    
            cursor.execute("""
                INSERT OR IGNORE INTO reviews (
                review_id, user_id, book_id, rating, review_text
                )
                VALUES (?, ?, ?, ?, ?)
            """, (
                rev['review_id'],
                rev['user_id'],
                rev["book_id"],
                safe_float(rev["rating"]),
                rev["review_text"]
            ))
            if i % 5000 == 0 and i > 0:
                conn.commit()
    conn.commit()

In [33]:
def load_json_ratings(path):
    data = []
    with gzip.open(path, "rt") as f:
        for i, line in enumerate(tqdm(f)):
            rat = json.loads(line)

            # only keep real ratings that do not include text (these are in reviews)
            if rat['is_read'] == True and rat['rating'] > 0 and rat['review_text_incomplete'] == '': 
                cursor.execute("""
                    INSERT OR IGNORE INTO ratings (
                    review_id, user_id, book_id, rating
                    )
                    VALUES (?, ?, ?, ?)
                """, (
                    rat['review_id'],
                    rat['user_id'],
                    rat["book_id"],
                    safe_float(rat["rating"])
                ))
            if i % 5000 == 0 and i > 0:
                conn.commit()
    conn.commit()

logic sanity check below for filtering ratings

In [32]:
path = os.path.join(DIR_GENRE, "goodreads_interactions_romance.json.gz")
with gzip.open(path, "rt") as f:
        for i, line in enumerate(tqdm(f)):
            rat = json.loads(line)
            if rat['is_read'] == True and rat['rating'] > 0 and rat['review_text_incomplete'] == '':
                print(rat)
            if i > 100:
                break

101it [00:00, 46116.34it/s]

{'user_id': '8842281e1d1347389f2ab93d60773d4d', 'book_id': '18135', 'review_id': 'd0aa055c0b0a5b0c4f13f9d358d51cd9', 'is_read': True, 'rating': 4, 'review_text_incomplete': '', 'date_added': 'Tue Jun 19 00:40:24 -0700 2007', 'date_updated': 'Wed Mar 22 11:45:33 -0700 2017', 'read_at': '', 'started_at': ''}
{'user_id': '72fb0d0087d28c832f15776b0d936598', 'book_id': '1885', 'review_id': '11beb5d3be4ebe9983ffa224b968dd17', 'is_read': True, 'rating': 5, 'review_text_incomplete': '', 'date_added': 'Mon Jun 04 19:01:18 -0700 2012', 'date_updated': 'Mon Jun 04 19:01:18 -0700 2012', 'read_at': '', 'started_at': ''}
{'user_id': 'ab2923b738ea3082f5f3efcbbfacb218', 'book_id': '47401', 'review_id': '303e3bb8890afde2b3849e9c1ebeaafe', 'is_read': True, 'rating': 4, 'review_text_incomplete': '', 'date_added': 'Wed Nov 11 12:45:11 -0800 2009', 'date_updated': 'Wed Nov 11 12:45:11 -0800 2009', 'read_at': '', 'started_at': ''}
{'user_id': 'ab2923b738ea3082f5f3efcbbfacb218', 'book_id': '227443', 'review_

## Load data into romance.db

In [7]:
# start with authors so they exist when we populate book_authors
# need to use the complete data because there is no genre specific author tables
DIR = './complete/'
load_json_authors(os.path.join(DIR, "goodreads_book_authors.json.gz"))

829529it [00:09, 91651.67it/s] 


In [14]:
# define the genre dir
DIR_GENRE = './byGenre/'

In [9]:
# load book data
load_json_books(os.path.join(DIR_GENRE, "goodreads_books_romance.json.gz"))

335449it [06:06, 914.72it/s] 


In [10]:
# load text reviews
load_json_reviews(os.path.join(DIR_GENRE, "goodreads_reviews_romance.json.gz"))

3565378it [03:58, 14939.43it/s]


In [34]:
# load ratings
load_json_ratings(os.path.join(DIR_GENRE, "goodreads_interactions_romance.json.gz"))

42792856it [53:34, 13311.04it/s]


## Derive Users Table

In [5]:
# create users table 

# first, create indexes on user_id to speed up aggregation
cursor.execute("""
CREATE INDEX idx_reviews_user ON reviews(user_id);
""")

cursor.execute("""
CREATE INDEX idx_ratings_user ON ratings(user_id);
""")

conn.commit()

# second, derive counts with pre-aggregation of reviews and ratings separately
cursor.execute("""
CREATE TABLE users AS

SELECT
    u.user_id,
    COALESCE(r.text_review_count, 0) AS text_review_count,
    COALESCE(rt.rating_count, 0) AS rating_count,
    COALESCE(r.text_review_count, 0) + COALESCE(rt.rating_count, 0) AS total_ratings,
    r.avg_text_rating,
    rt.avg_rating_only

FROM (
    SELECT user_id FROM reviews
    UNION
    SELECT user_id FROM ratings
) u

LEFT JOIN (
    SELECT
        user_id,
        COUNT(*) AS text_review_count,
        AVG(rating) AS avg_text_rating
    FROM reviews
    GROUP BY user_id
) r USING (user_id)

LEFT JOIN (
    SELECT
        user_id,
        COUNT(*) AS rating_count,
        AVG(rating) AS avg_rating_only
    FROM ratings
    GROUP BY user_id
) rt USING (user_id);

""")
conn.commit()


# Below crashed without pre-aggregation due to both tables being millions of rows
# cursor.execute("""
# CREATE TABLE users AS
# SELECT
#     u.user_id,

#     -- counts
#     COUNT(DISTINCT r.review_id) AS text_review_count,
#     COUNT(DISTINCT rt.review_id) AS rating_count,
#     COUNT(DISTINCT r.review_id) + COUNT(DISTINCT rt.review_id) AS total_ratings,

#     -- averages
#     AVG(r.rating) AS avg_text_review_rating,
#     AVG(rt.rating) AS avg_rating_only

# FROM (
#     -- union all users from both tables
#     SELECT user_id FROM reviews
#     UNION
#     SELECT user_id FROM ratings
# ) u

# LEFT JOIN reviews r
#     ON u.user_id = r.user_id

# LEFT JOIN ratings rt
#     ON u.user_id = rt.user_id

# GROUP BY u.user_id;
# """)
# conn.commit()

## Generate ER diagram

In [8]:
from eralchemy2 import render_er

# generate ER diagram 
render_er("sqlite:///byGenre/romance.db", "romance_ER_diagram.png")

## Sanity Check + Table Summary

Make sure all looks good, calculate some summary stats

In [11]:
# check book table
df = pd.read_sql_query(
    "SELECT * FROM books",
    conn
)

df.head()

,book_id,title,description,average_rating,ratings_count,text_reviews_count,num_pages,publisher,publication_year,publication_month,publication_day
0,34883016,Playmaker: A Venom Series Novella,Secrets. Sometimes keeping them in confidence ...,3.86,5,4,NaN,Gone Writing Publishing,2017.0,5.0,3.0
1,29074693,"Prowled Darkness (Dante's Circle, #7)",,4.23,149,21,NaN,,NaN,NaN,NaN
2,3209316,Emma,The funny and heartwarming story of a young la...,3.99,42,8,544.0,Brilliance Audio,2005.0,9.0,25.0
3,30838933,"Guardian Cougar (Finding Fatherhood, #2)","In the Finding Fatherhood series, these shifte...",4.31,139,27,NaN,,NaN,NaN,NaN
4,27419760,Wedding Girl,You've Got Mailmeets Julie & Juliain the new f...,3.98,167,15,NaN,,NaN,NaN,NaN


In [12]:
print(len(df))
df['description'] = df['description'].replace('', np.nan)
df.isnull().sum()

335449


book_id                    0
title                      0
description            30612
average_rating             0
ratings_count              0
text_reviews_count         0
num_pages             136630
publisher                  0
publication_year      101800
publication_month     114525
publication_day       132443
dtype: int64

### Books Summary:
N = 335,449 books  
30,612 have no description (\~10%)  
136,630 have no page counts (\~40%)  
upto 114,525 have missing dates (\~35%)  

In [9]:
# check book_authors table
df = pd.read_sql_query(
    "SELECT * FROM book_authors",
    conn
)

df.head()

,book_id,author_id
0,34883016,5807700
1,29074693,5360266
2,3209316,1265
3,30838933,90411
4,30838933,14356708


In [10]:
print(len(df))
df.isnull().sum()

400831


book_id      0
author_id    0
dtype: int64

### book-authors Summary:
N = 400,831 book-author connections  
3,108 have null text  

In [34]:
# check authors table
df = pd.read_sql_query(
    "SELECT * FROM authors",
    conn
)

df.head()

,author_id,author_name,author_average_rating,author_ratings_count
0,604031,Ronald J. Fields,3.98,49
1,626222,Anita Diamant,4.08,546796
2,10333,Barbara Hambly,3.92,122118
3,9212,Jennifer Weiner,3.68,888522
4,149918,Nigel Pennick,3.82,1740


In [35]:
print(len(df))
df.isnull().sum()

829529


author_id                0
author_name              0
author_average_rating    0
author_ratings_count     0
dtype: int64

### Authors Summary:
N = 829,529 authors  
no missing values    

In [36]:
# check shelves table
df = pd.read_sql_query(
    "SELECT * FROM shelves",
    conn
)

df.head()

,shelf_id,shelf_name
0,1,to-read
1,2,ibooks
2,3,favorite-authors
3,4,may-2017-dr-reads
4,5,sports-romance


In [37]:
print(len(df))
df.isnull().sum()

464952


shelf_id      0
shelf_name    0
dtype: int64

### Shelves Summary:
N = 464,952 shelves  
no missing values    


In [36]:
# check ratings table
df = pd.read_sql_query(
    "SELECT * FROM ratings",
    conn
)

df.head()

,review_id,user_id,book_id,rating
0,d0aa055c0b0a5b0c4f13f9d358d51cd9,8842281e1d1347389f2ab93d60773d4d,18135,4.0
1,11beb5d3be4ebe9983ffa224b968dd17,72fb0d0087d28c832f15776b0d936598,1885,5.0
2,303e3bb8890afde2b3849e9c1ebeaafe,ab2923b738ea3082f5f3efcbbfacb218,47401,4.0
3,fa6ede0afb0a5e0b4d7b78855f81d8ed,ab2923b738ea3082f5f3efcbbfacb218,227443,3.0
4,d6d8cb411ba38957e6c9f0fd80e7617c,ab2923b738ea3082f5f3efcbbfacb218,6185,3.0


In [37]:
print(len(df))
df.isnull().sum()

16231881


review_id    0
user_id      0
book_id      0
rating       0
dtype: int64

In [38]:
sum(df['rating'] == 0)

0

### Ratings Summary:
N = 16,231,881 ratings  
No missing values  
No 0 star ratings  

In [55]:
# check ratings table
df = pd.read_sql_query(
    "SELECT * FROM reviews",
    conn
)

df.head()

,review_id,user_id,book_id,rating,review_text
0,5347a776a1703b823ce029d68ae98275,8842281e1d1347389f2ab93d60773d4d,1893,5.0,** spoiler alert ** \n So the other day Elizab...
1,719711cc71eec0bb54d2d97322c0e11b,72fb0d0087d28c832f15776b0d936598,17939501,5.0,"It is very hard to believe this is all true, b..."
2,6a870a66f732183b60214d57fa553093,72fb0d0087d28c832f15776b0d936598,15706923,2.0,Ehhhhhh. \n Really nothing to rave about. It w...
3,f73a70f64564d4a8f4cfb2d2e9d5836f,72fb0d0087d28c832f15776b0d936598,7840190,4.0,Enjoyable read! I liked that Connie is not a t...
4,547aeff3c7ee5b4a39a93cd2a720b001,72fb0d0087d28c832f15776b0d936598,15463724,4.0,There are definitely too many books lately wit...


In [56]:
df['review_text'] = df['review_text'].replace('', np.nan)
df.isnull().sum()

review_id         0
user_id           0
book_id           0
rating            0
review_text    3108
dtype: int64

In [57]:
len(df)

3565378

### Reviews Summary:
N = 3,565,378 text reviews  
3,108 have null text  

In [6]:
# check ratings table
df = pd.read_sql_query(
    "SELECT * FROM users",
    conn
)

df.head()

,user_id,text_review_count,rating_count,total_ratings,avg_text_rating,avg_rating_only
0,00000377eea48021d3002730d56aca9a,0,2,2,NaN,4.000000
1,00004584d524ec468619e81b176cc991,0,21,21,NaN,4.047619
2,000079c580bbe45e1500acabe551b276,0,1,1,NaN,2.000000
3,00009e46d18f223a82b22da38586b605,2,1,3,3.0,4.000000
4,0000bad9195b66484e98f7f4be5f227a,0,6,6,NaN,3.166667


In [7]:
print(len(df))
df.isnull().sum()

569320


user_id                   0
text_review_count         0
rating_count              0
total_ratings             0
avg_text_rating      371179
avg_rating_only       20042
dtype: int64

### Users Summary:
N = 569,320 users  
371,179 have only ratings, no text reviews  
20,042 have only text reviews, no ratings  